# MGMT298D: Science and Strategy of AI
### Week 6 - Convolutional Neural Networks
### Application: Object Recognition

## Import Libraries and Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras import layers
from keras.utils import plot_model
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Load CIFAR-10 dataset (32x32 color images)
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

print(f"Training: {x_train.shape} | Test: {x_test.shape}")

## Visualize Sample Images

In [ ]:
plt.figure(figsize=(10, 5))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i])
    plt.title(CLASS_NAMES[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Model A: MLP Baseline

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True)

def plot_history(history, title):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['accuracy'], label='Train')
    ax1.plot(history.history['val_accuracy'], label='Validation')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
    ax2.plot(history.history['loss'], label='Train')
    ax2.plot(history.history['val_loss'], label='Validation')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
    fig.suptitle(title)
    plt.show()

model_A = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Flatten(),
    layers.Dense(256, activation="relu"),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_A.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
plot_model(model_A, show_shapes=True, show_layer_activations=True)

history_A = model_A.fit(x_train, y_train, epochs=50, batch_size=128,
                        validation_split=0.15, callbacks=[early_stopping], verbose=1)

test_loss_A, test_acc_A = model_A.evaluate(x_test, y_test, verbose=0)
print(f"Model A (MLP) — Test Accuracy: {test_acc_A*100:.2f}%")
plot_history(history_A, "Model A: MLP Baseline")

## Model B: Simple CNN

In [ ]:
model_B = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(10, activation="softmax")
])
model_B.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
plot_model(model_B, show_shapes=True, show_layer_activations=True)

history_B = model_B.fit(x_train, y_train, epochs=50, batch_size=128,
                        validation_split=0.15, callbacks=[early_stopping], verbose=1)

test_loss_B, test_acc_B = model_B.evaluate(x_test, y_test, verbose=0)
print(f"Model B (Simple CNN) — Test Accuracy: {test_acc_B*100:.2f}%")
plot_history(history_B, "Model B: Simple CNN")

## Model C: Advanced CNN with Regularization

In [ ]:
model_C = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Conv2D(32, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.Conv2D(64, 3, padding="same", use_bias=False),
    layers.BatchNormalization(),
    layers.ReLU(),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),
    
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(10, activation="softmax")
])
model_C.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
plot_model(model_C, show_shapes=True, show_layer_activations=True)

history_C = model_C.fit(x_train, y_train, epochs=50, batch_size=128,
                        validation_split=0.15, callbacks=[early_stopping], verbose=1)

test_loss_C, test_acc_C = model_C.evaluate(x_test, y_test, verbose=0)
print(f"Model C (Advanced CNN) — Test Accuracy: {test_acc_C*100:.2f}%")
plot_history(history_C, "Model C: Advanced CNN")

## Pre-trained Models: ResNet50 and MobileNetV2

In [ ]:
from tensorflow.keras.applications import resnet50, mobilenet_v2
from tensorflow.keras.preprocessing import image
from IPython.display import display, HTML
import ipywidgets as widgets
import io
from PIL import Image

# Load pre-trained models (trained on ImageNet: 1M+ images, 1000 classes)
resnet_model = resnet50.ResNet50(weights="imagenet")
mobilenet_model = mobilenet_v2.MobileNetV2(weights="imagenet")
print("Pre-trained models loaded")

def predict_image(img_bytes):
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    img_resized = img.resize((224, 224))
    img_array = image.img_to_array(img_resized)
    
    # MobileNetV2 predictions
    mobilenet_input = mobilenet_v2.preprocess_input(np.expand_dims(img_array.copy(), axis=0))
    mobilenet_preds = mobilenet_model.predict(mobilenet_input)
    decoded_mobilenet = mobilenet_v2.decode_predictions(mobilenet_preds, top=5)[0]
    
    # ResNet50 predictions
    resnet_input = resnet50.preprocess_input(np.expand_dims(img_array.copy(), axis=0))
    resnet_preds = resnet_model.predict(resnet_input)
    decoded_resnet = resnet50.decode_predictions(resnet_preds, top=5)[0]
    
    return img, decoded_mobilenet, decoded_resnet

# File upload widget
uploader = widgets.FileUpload(accept='image/*', multiple=False)
output_area = widgets.Output()

def on_upload(change):
    if not change['new']:
        return
    file_content = list(uploader.value.values())[0]['content']
    img, mobilenet_top5, resnet_top5 = predict_image(file_content)
    
    with output_area:
        output_area.clear_output()
        display(img.resize((200, 200)))
        print("\nMobileNetV2 Top 5:")
        for _, name, score in mobilenet_top5:
            print(f"  {name}: {score:.2%}")
        print("\nResNet50 Top 5:")
        for _, name, score in resnet_top5:
            print(f"  {name}: {score:.2%}")

uploader.observe(on_upload, names='_counter')
display(HTML("<b>Upload an image to classify:</b>"))
display(uploader, output_area)

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['A (MLP)', 'B (Simple CNN)', 'C (Advanced CNN)'],
    'Test Accuracy': [test_acc_A, test_acc_B, test_acc_C],
    'Parameters': [model_A.count_params(), model_B.count_params(), model_C.count_params()]
})
results['Test Accuracy'] = results['Test Accuracy'] * 100
results = results.sort_values(by='Test Accuracy', ascending=False).set_index('Model')
print(results.to_string(formatters={'Test Accuracy': '{:.2f}%'.format, 'Parameters': '{:,}'.format}))